# MAPL+ Pipeline — Step 1 to Step 3

| Step | Deskripsi | Referensi |
|------|-----------|-----------|
| 1 | Aggregate raw transactions → daily panel | — |
| 2 | Own-price elasticity per SKU (log-log OLS) | OpenStax Introductory Business Statistics, Ch. 13.5 |
| 3 | Cannibalization detection via Difference-in-Differences | Van Heerde et al. (2004), McColl et al. (2020), Varian (2016), Herrala (2018) |

## Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


---
## Step 1 — Aggregate ke Daily Panel

Aggregate raw transactions ke daily panel.  
Satu baris = **1 SKU × 1 hari × 1 branch**.

> Oktober (bulan 10) di-exclude sesuai preprocessing awal.

In [2]:
def step1_daily_panel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate raw transactions ke daily panel.
    Satu baris = 1 SKU × 1 hari × 1 branch.
    """
    # ← Hapus pd.read_csv dan pd.concat di sini, gunakan df parameter langsung

    df = df.copy()  # hindari modifikasi df asli
    df['Date'] = pd.to_datetime(df['Date'])

    # Exclude Oktober (bulan 10) sesuai preprocessing awal
    df = df[df['Date'].dt.month != 10]

    # Tambah WeekNum (0-indexed dari hari pertama)
    min_date = df['Date'].min()
    df['WeekNum'] = ((df['Date'] - min_date).dt.days // 7).astype(int)

    daily = (
        df.groupby(['Date', 'WeekNum', 'Branch', 'SKU_ID', 'SKU', 'Brand', 'SKU_Category'])
        .agg(
            DailyQty    = ('Qty',               'sum'),
            Price       = ('DiscountedPrice',   'first'),
            Discount    = ('DiscountPercentage','first'),
            IsPromo     = ('IsPromo',           'max'),
            NormalPrice = ('NormalPrice',       'first'),
        )
        .reset_index()
    )

    print(f'[Step 1] Daily panel: {len(daily):,} baris '
          f'({daily["Date"].nunique()} hari × '
          f'{daily["Branch"].nunique()} branch × '
          f'{daily["SKU_ID"].nunique()} SKU)')
    return daily

In [3]:
# ── Opsi A: mulai dari raw transactions ──
df1 = pd.read_csv('transaction_1_v5.csv')
df2 = pd.read_csv('transaction_2_v5.csv')
df  = pd.concat([df1, df2], ignore_index=True)
if 'Date' not in df.columns:
    df['Date'] = pd.to_datetime(df['DateTime']).dt.date
    df['Date'] = pd.to_datetime(df['Date'])
print(f'Raw data loaded: {len(df):,} rows\n')
daily = step1_daily_panel(df)
daily.to_csv('daily_panel.csv', index=False)
print('→ Saved: daily_panel.csv')

# ── Opsi B: daily panel sudah ada (skip step 1) ──
# DAILY_PATH = 'daily_panel.csv'  # ganti sesuai path
# daily = pd.read_csv(DAILY_PATH, parse_dates=['Date'])
# print(f'Daily panel loaded: {len(daily):,} rows')
# daily

Raw data loaded: 205,946 rows

[Step 1] Daily panel: 12,240 baris (153 hari × 4 branch × 20 SKU)
→ Saved: daily_panel.csv


---
## Step 2 — Own-Price Elasticity per SKU

Estimasi own-price elasticity per SKU menggunakan **log-log OLS**.

$$\ln(\text{DailyQty}) = \alpha + \beta \cdot \ln(\text{Price}) + \varepsilon$$

$\beta$ = own-price elasticity *(OpenStax Ch. 13.5, Case 4: log-log)*

Pooled across all 4 branches per SKU.

In [4]:
def step2_own_elasticity(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Estimasi own-price elasticity per SKU menggunakan log-log OLS.

    Model: ln(DailyQty) = α + β × ln(Price) + ε
    β = own-price elasticity (OpenStax Ch. 13.5, Case 4: log-log)

    Pooled across all 4 branches per SKU.
    """
    results = []

    for sku_id, grp in daily.groupby('SKU_ID'):
        grp = grp.copy()

        # Filter: harga dan qty harus > 0
        grp = grp[(grp['Price'] > 0) & (grp['DailyQty'] > 0)]
        if len(grp) < 30:
            continue

        grp['ln_qty']   = np.log(grp['DailyQty'])
        grp['ln_price'] = np.log(grp['Price'])

        try:
            model    = ols('ln_qty ~ ln_price', data=grp).fit()
            beta     = model.params['ln_price']
            pval     = model.pvalues['ln_price']
            r2       = model.rsquared
            sku_name = grp['SKU'].iloc[0]

            results.append({
                'SKU_ID':        sku_id,
                'SKU':           sku_name,
                'OwnElasticity': round(beta, 4),
                'p_value':       round(pval, 4),
                'R2':            round(r2, 4),
                'N_obs':         len(grp),
                'Source':        'ols_loglog',
            })
        except Exception as e:
            print(f'  [Step 2] SKU {sku_id} error: {e}')

    elasticity_df = pd.DataFrame(results)
    print(f'\n[Step 2] Elasticity estimated for {len(elasticity_df)} SKUs')
    return elasticity_df

In [5]:
elasticity_df = step2_own_elasticity(daily)

elasticity_df.to_csv('elasticity_per_sku.csv', index=False)
print('→ Saved: elasticity_per_sku.csv')

elasticity_df[['SKU_ID', 'SKU', 'OwnElasticity', 'p_value', 'R2']]


[Step 2] Elasticity estimated for 20 SKUs
→ Saved: elasticity_per_sku.csv


,SKU_ID,SKU,OwnElasticity,p_value,R2
0,S001,Richeese Wafer Keju 50g,-1.4360,0.0000,0.0338
1,S002,Richoco Wafer Cokelat 50g,-0.6261,0.0022,0.0153
2,S003,Richeese Wafer Keju 10g Renceng,-1.0113,0.0002,0.0230
3,S004,Richoco Wafer Cokelat 10g Renceng,-1.1112,0.0000,0.0406
4,S005,Nextar Brownies Pie 40g,-1.1361,0.0000,0.0460
5,S006,Nextar Nastar Pie 30g,-0.9365,0.0000,0.0320
6,S007,Richeese Siip Keju 20g,-0.8255,0.0000,0.0273
7,S008,Richoco Ahh! Extruded 15g,-0.5087,0.0781,0.0051
8,S009,Richeese Mi Instan Keju Pedas,-0.8243,0.0005,0.0197
9,S010,Richeese Mi Instan Ramen Keju,-0.5369,0.0076,0.0116


---
## Step 3 — Cannibalization Detection (DiD)

Deteksi cannibalization antar SKU menggunakan **DiD estimator** (Varian, 2016).

$$\text{DiD}_{AB} = \overline{\text{Residual}_B \mid \text{IsPromo}_A=1} - \overline{\text{Residual}_B \mid \text{IsPromo}_A=0}$$

**Cannibalization coefficient** (Herrala, 2018):

$$C(A \to B) = \frac{|\text{DiD}_{AB}|}{\overline{\text{Uplift}_A \mid \text{IsPromo}_A=1}}$$

Cannibalization **confirmed** jika: $p < p_{\text{threshold}}$ **AND** $\text{DiD}_{AB} < 0$

In [6]:
def step3_cannibalization(daily: pd.DataFrame,
                           p_threshold: float = 0.05,
                           min_promo_days: int = 10,
                           max_coef: float = 1.0) -> pd.DataFrame:
    """
    Deteksi cannibalization antar SKU menggunakan DiD estimator.

    Framework (Varian, 2016):
      DiD_AB = mean(Residual_B | IsPromo_A=1)
             - mean(Residual_B | IsPromo_A=0)

    Cannibalization coefficient (Herrala, 2018):
      C(A→B) = |DiD_AB| / mean(Uplift_A | IsPromo_A=1)
        → di-cap di max_coef (default 1.0) untuk menghindari
        rasio yang tidak masuk akal secara bisnis

    Cannibalization confirmed jika: p < p_threshold AND DiD_AB < 0

    NOTE: Baseline SKU B difit dari hari di mana KEDUA SKU A dan B
    sama-sama tidak promo (intersection), bukan hanya dari hari A
    tidak promo. Ini mencegah own-price effect SKU B sendiri
    mendistorsi baseline-nya.
    """
    skus     = daily['SKU_ID'].unique()
    branches = daily['Branch'].unique()
    records  = []

    for branch in branches:
        branch_data = daily[daily['Branch'] == branch].copy()

        for sku_a in skus:
            # Ambil hari-hari promo SKU A di branch ini
            a_data = branch_data[branch_data['SKU_ID'] == sku_a][
                ['Date', 'WeekNum', 'IsPromo', 'DailyQty']
            ].copy()

            promo_days_a    = set(a_data[a_data['IsPromo'] == 1]['Date'])
            no_promo_days_a = set(a_data[a_data['IsPromo'] == 0]['Date'])

            if len(promo_days_a) < min_promo_days:
                continue  # Tidak cukup promo days untuk SKU A ini

            for sku_b in skus:
                if sku_a == sku_b:
                    continue

                b_data = branch_data[branch_data['SKU_ID'] == sku_b][
                    ['Date', 'WeekNum', 'IsPromo', 'DailyQty']
                ].copy()

                if len(b_data) < 30:
                    continue

                # ── Sub-step 3A: Baseline SKU B ──
                # Counterfactual: demand B "seandainya tidak ada promosi A
                # DAN tidak ada promosi B sendiri" (clean baseline)
                # (Varian 2016: "counterfactual constructed using data
                #  from before/outside the treatment period")
                no_promo_days_b = set(b_data[b_data['IsPromo'] == 0]['Date'])

                # Hari "bersih": A tidak promo DAN B tidak promo
                clean_days = no_promo_days_a & no_promo_days_b

                b_no_promo = b_data[b_data['Date'].isin(clean_days)].copy()

                if len(b_no_promo) < 10:
                    continue

                try:
                    baseline_model = ols(
                        'DailyQty ~ WeekNum', data=b_no_promo
                    ).fit()
                except Exception:
                    continue

                # Predict baseline untuk semua hari
                b_data = b_data.copy()
                b_data['BaselineDemand'] = baseline_model.predict(b_data)
                b_data['Residual_B']     = b_data['DailyQty'] - b_data['BaselineDemand']

                # ── Sub-step 3B: DiD Estimator ──
                # Treatment: hari SKU A promo (B tidak promo) → lihat Residual B
                # Control  : hari SKU A & B tidak promo        → lihat Residual B
                #
                # Catatan: treatment tetap pakai semua hari A promo,
                # TIDAK difilter B harus tidak promo, supaya tetap
                # menangkap interaksi nyata saat A promo (termasuk
                # kasus B kebetulan ikut promo bersamaan)
                residual_treatment = b_data[
                    b_data['Date'].isin(promo_days_a)
                ]['Residual_B'].dropna()

                residual_control = b_data[
                    b_data['Date'].isin(clean_days)
                ]['Residual_B'].dropna()

                if len(residual_treatment) < 5 or len(residual_control) < 5:
                    continue

                did_ab = residual_treatment.mean() - residual_control.mean()

                # ── Sub-step 3C: Significance test ──
                # H0: DiD_AB = 0, H1: DiD_AB < 0 (one-sided)
                _, p_one = stats.ttest_ind(
                    residual_treatment, residual_control,
                    equal_var=False, alternative='less'
                )
                # scipy "less" = H1: mean(treatment) < mean(control) — already one-sided

                # Cannibalization coefficient (Herrala, 2018)
                # C(A→B) = |DiD_AB| / mean(Uplift_A saat promo)
                a_promo_data = a_data[a_data['Date'].isin(promo_days_a)].copy()
                a_clean      = a_data[a_data['Date'].isin(clean_days)].copy()

                cannib_coef = np.nan
                raw_coef    = np.nan

                if len(a_promo_data) > 0 and len(a_clean) > 0:
                    mean_uplift_a = (
                        a_promo_data['DailyQty'].mean() -
                        a_clean['DailyQty'].mean()
                    )
                    if mean_uplift_a > 0:
                        raw_coef    = abs(did_ab) / mean_uplift_a
                        cannib_coef = min(raw_coef, max_coef)  # cap di 1.0

                records.append({
                    "Branch":       branch,
                    'SKU_A':        sku_a,   # promotor
                    'SKU_B':        sku_b,   # potential victim
                    "DiD_AB":       round(did_ab, 4),
                    "p_value":      round(p_one, 4),
                    "Cannib_Coef":  round(cannib_coef, 4) if not np.isnan(cannib_coef) else np.nan,
                    "Confirmed":    (p_one < p_threshold) and (did_ab < 0),
                    "IsCapped":     (not np.isnan(raw_coef)) and (raw_coef > max_coef),
                    "N_treatment":  len(residual_treatment),
                    "N_control":    len(residual_control),
                })

    cannib_df = pd.DataFrame(records)

    confirmed = cannib_df[cannib_df['Confirmed'] == True]
    
    print(f'\n[Step 3] Total pairs tested       : {len(cannib_df):,}')
    print(f'[Step 3] Cannibalization confirmed: {len(confirmed)} pairs '
          f'(p < {p_threshold}, DiD < 0)')
    print(f"[Step 3] Cannib_Coef capped at    : {max_coef}")

    return cannib_df

In [7]:
cannib_df = step3_cannibalization(daily, p_threshold=0.05, min_promo_days=10)

cannib_df.to_csv('cannibalization_detail.csv', index=False)
print('→ Saved: cannibalization_detail.csv')

cannib_df[cannib_df['Confirmed'] == True].head(20)


[Step 3] Total pairs tested       : 1,520
[Step 3] Cannibalization confirmed: 76 pairs (p < 0.05, DiD < 0)
[Step 3] Cannib_Coef capped at    : 1.0
→ Saved: cannibalization_detail.csv


,Branch,SKU_A,SKU_B,DiD_AB,p_value,Cannib_Coef,Confirmed,IsCapped,N_treatment,N_control
39,Bandung,S003,S002,-1.9717,0.0137,0.2640,True,False,39,77
40,Bandung,S003,S004,-2.4402,0.0129,0.3065,True,False,39,78
44,Bandung,S003,S008,-3.3784,0.0008,0.4131,True,False,39,84
99,Bandung,S006,S005,-2.2557,0.0221,0.4355,True,False,50,66
104,Bandung,S006,S011,-1.8637,0.0158,0.3529,True,False,50,79
131,Bandung,S007,S019,-2.6942,0.0035,0.3286,True,False,51,72
160,Bandung,S009,S010,-2.0705,0.0230,0.4407,True,False,39,88
170,Bandung,S009,S020,-2.4485,0.0042,0.4768,True,False,39,74
229,Bandung,S013,S002,-1.8423,0.0129,0.2806,True,False,38,79
242,Bandung,S013,S016,-1.7811,0.0427,0.3725,True,False,38,77


---
## Cannibalization Matrix

Build **N×N matrix** dari hasil Step 3.  
`C[A][B]` = koefisien cannibalization SKU A terhadap SKU B, dirata-rata across branches.  
Hanya pair yang **confirmed** yang non-zero.

In [8]:
def build_cannib_matrix(cannib_df: pd.DataFrame,
                         skus: list,
                         agg: str = 'mean') -> pd.DataFrame:
    """
    Build NxN cannibalization matrix dari hasil step3.
    C[A][B] = koefisien cannibalization SKU A terhadap SKU B,
              averaged across branches.
    Hanya pair yang confirmed yang non-zero.
    """
    confirmed = cannib_df[cannib_df['Confirmed'] == True].copy()

    if agg == 'mean':
        agg_df = (
            confirmed.groupby(['SKU_A', 'SKU_B'])['Cannib_Coef']
            .mean()
            .reset_index()
        )
    else:
        agg_df = (
            confirmed.groupby(['SKU_A', 'SKU_B'])['Cannib_Coef']
            .median()
            .reset_index()
        )

    matrix = pd.DataFrame(0.0, index=skus, columns=skus)
    for _, row in agg_df.iterrows():
        if row['SKU_A'] in skus and row['SKU_B'] in skus:
            matrix.loc[row['SKU_A'], row['SKU_B']] = round(row['Cannib_Coef'], 4)

    return matrix

In [9]:
skus   = sorted(daily['SKU_ID'].unique())
matrix = build_cannib_matrix(cannib_df, skus, agg='mean')

matrix.to_csv('cannibalization_matrix.csv')
print('→ Saved: cannibalization_matrix.csv')
print(f'\nCannibalization matrix ({len(skus)}×{len(skus)}):')
matrix

→ Saved: cannibalization_matrix.csv

Cannibalization matrix (20×20):


,S001,S002,S003,S004,S005,S006,S007,S008,S009,S010,S011,S012,S013,S014,S015,S016,S017,S018,S019,S020
S001,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2738,0.0000
S002,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.7041,0.0000,0.0000,0.0000,0.0000
S003,0.0,0.2640,0.0000,0.2782,0.0000,0.0000,0.0000,0.4131,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.3373,0.0000,0.0000,0.0000,0.0000,0.0000
S004,0.0,0.0000,0.0000,0.0000,0.0000,0.4295,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.4249,0.1806,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
S005,0.0,0.0000,0.0000,0.0000,0.0000,0.2436,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
S006,0.0,0.0000,0.0000,0.0000,0.4788,0.0000,0.0000,0.0000,0.0000,0.0000,0.3529,0.0,0.0000,0.0000,0.3645,0.0000,0.0000,0.0000,0.5956,0.0000
S007,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3242,0.0000
S008,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
S009,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4574,0.0000,0.0,0.0000,0.0000,0.0000,0.5111,0.1912,0.0000,0.0000,0.5456
S010,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.6912,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [11]:
print(cannib_df['Confirmed'].sum())
print(len(cannib_df))

76
1520


In [12]:
# Bandingin rata-rata N_control sebelum vs sesudah
print("N_control rata-rata:", cannib_df['N_control'].mean())
print("N_control minimum  :", cannib_df['N_control'].min())

N_control rata-rata: 81.0
N_control minimum  : 64


---
✅ **Step 1–3 selesai.**

Output files:
- `daily_panel.csv` — daily panel (Step 1)
- `elasticity_per_sku.csv` — own-price elasticity per SKU (Step 2)
- `cannibalization_detail.csv` — detail DiD per pair per branch (Step 3)
- `cannibalization_matrix.csv` — N×N cannibalization matrix (Step 3)

In [10]:
# Lihat slope baseline S002 di salah satu branch
import numpy as np
from statsmodels.formula.api import ols

# Ambil S002 Bandung, no-promo days saja
s002_bandung = daily[(daily["SKU_ID"] == "S002") & 
                     (daily["Branch"] == "Bandung")].copy()

no_promo = s002_bandung[s002_bandung["IsPromo"] == 0]

model = ols("DailyQty ~ WeekNum", data=no_promo).fit()
print("α (intercept):", round(model.params["Intercept"], 2))
print("β (slope)    :", round(model.params["WeekNum"], 2))

baseline_week0  = model.params["Intercept"] + model.params["WeekNum"] * 0
baseline_week21 = model.params["Intercept"] + model.params["WeekNum"] * 21

print(f"\nBaseline Week 0  : {baseline_week0:.2f}")
print(f"Baseline Week 21 : {baseline_week21:.2f}")
print(f"Selisih          : {baseline_week21 - baseline_week0:.2f}")
print(f"Selisih %        : {((baseline_week21-baseline_week0)/baseline_week0*100):.1f}%")

α (intercept): 20.21
β (slope)    : 0.03

Baseline Week 0  : 20.21
Baseline Week 21 : 20.91
Selisih          : 0.70
Selisih %        : 3.4%
